# Embedding SigLIP2 + OCR cho keyframe AIC trên Kaggle

Chạy trên keyframe **đã trích sẵn** (dataset Kaggle), không trích lại. Ra 2 thứ để mang về máy:

| File | Dùng ở máy |
|---|---|
| `siglip2_features.zip` → `<vid>.npy` | `build_index.py --feat-dir ...` → index SigLIP2 |
| `ocr.zip` → `<vid>.jsonl` | `app.py --ocr ...` → ô lọc OCR |

**Chạy 2 notebook cùng lúc (nhanh gấp đôi)**: import notebook này 2 lần thành 2 notebook riêng, cả hai cùng
`N_NOTEBOOKS = 2`; notebook thứ nhất `NOTEBOOK_ID = 0`, thứ hai `NOTEBOOK_ID = 1`. Mỗi notebook làm một nửa số video,
tải về `ocr_part0.zip` + `ocr_part1.zip` rồi giải nén **chung một thư mục**.

**Các bước**
1. **Add Input** → dataset keyframes (mỗi batch có `map-keyframes/` và `keyframes/`; tar hay đã giải nén đều được).
2. **Settings** → Accelerator **GPU T4 x2**, **Internet: On** (cần xác minh số điện thoại để bật).
3. **Chạy thử**: để `LIMIT = 2` → *Run All* → cell 6 in tốc độ (ảnh/s) và ETA của từng việc.
4. **Chạy thật**: `LIMIT = 0` → **Save Version → Save & Run All (Commit)**. Chạy nền, đóng trình duyệt được.
   Chỉ OCR: mỗi GPU chạy `OCR_PROCS_PER_GPU` tiến trình. Chỉ embedding: mỗi GPU 1 tiến trình. Bật cả hai: embedding GPU 0, OCR GPU 1.
5. Xong → tab **Output** của version đó → tải `ocr_part<ID>.zip` (và `siglip2_features_part<ID>.zip` nếu bật embedding).

**Chưa xong trong 1 lần** (tự dừng ở `MAX_HOURS` để không mất kết quả): *Add Input* → *Notebook Output* của version
vừa chạy → điền `RESUME_FEAT` / `RESUME_OCR` (cell 6 in sẵn đường dẫn) → Save & Run All lại. Video đã xong được bỏ qua.

In [ ]:
# 1. CAU HINH -- chi sua o cell nay
LIMIT = 0            # >0 = chi chay N video moi tien trinh (thu nhanh); 0 = chay het
RUN_EMBED = False    # embedding SigLIP2 da xong
RUN_OCR = True
NOTEBOOK_ID = 0      # notebook thu nhat = 0, notebook thu hai = 1
N_NOTEBOOKS = 2      # so notebook chay cung luc, moi cai lam 1 phan video
OCR_PROCS_PER_GPU = 2  # 2 tien trinh / GPU: tien trinh nay xu ly CPU thi tien trinh kia dung GPU
MAX_HOURS = 11.0     # Kaggle cat phien o 12h -> tu dung truoc de con luu ket qua

DATA_DIRS = []       # de trong = tu tim moi thu muc co map-keyframes/ trong /kaggle/input

EMBED_MODEL, EMBED_PRETRAINED = 'ViT-SO400M-16-SigLIP2-384', 'webli'
EMBED_BATCH = 64
OCR_MAG = 1.0        # 1.5 = bat chu nho (ticker, bien hieu) tot hon, cham ~2 lan
OCR_BATCH = 8        # 2 tien trinh chung 1 GPU T4 16GB -> lo nho cho an toan bo nho

OUT_FEAT = '/kaggle/working/features-vit-so400m-16-siglip2-384'
OUT_OCR = '/kaggle/working/ocr'
RESUME_FEAT = []     # chay tiep: ['/kaggle/input/<notebook-output>/features-vit-so400m-16-siglip2-384']
RESUME_OCR = []      # chay tiep: ['/kaggle/input/<notebook-output>/ocr']

In [ ]:
# 2. Cai thu vien (can Internet: On)
!nvidia-smi -L
!pip install -q -U open_clip_torch timm
!pip install -q easyocr

In [ ]:
%%writefile embed_keyframes.py
"""Encode lai keyframe DA TRICH bang model manh hon (SigLIP2, DFN ViT-H...), doc thang tu file tar.

Khong trich keyframe lai: dung nguyen keyframes/<vid>.tar + map-keyframes/<vid>.csv cua notebook.

Dau ra: <data>/features-<model>/<vid>.npy  float16, L2-norm, CUNG THU TU voi map-keyframes/<vid>.csv
-> build_index.py dung duoc ngay, khong phai sua:
    python build_index.py --data .../Batch1 .../Batch2 --out .../index_siglip2 \
        --feat-dir features-vit-so400m-16-siglip2-384 --model ViT-SO400M-16-SigLIP2-384 --pretrained webli

Checkpoint: video da co .npy dung so dong voi csv thi bo qua -> bi ngat thi chay lai dung lenh cu.
Chia nhieu tai khoan Colab/Kaggle: --shard 0/2 o may 1, --shard 1/2 o may 2 (ghi chung 1 thu muc Drive).

    pip install -U open_clip_torch timm
    python embed_keyframes.py --data /content/drive/MyDrive/Batch1 /content/drive/MyDrive/Batch2 --limit 2
    python embed_keyframes.py --data /content/drive/MyDrive/Batch1 /content/drive/MyDrive/Batch2

Kaggle (dataset chi doc): bat buoc --out /kaggle/working/...; --resume <output lan truoc> de chay tiep,
--max-hours 11 de tu dung truoc gioi han 12h ma van giu ket qua. Xem kaggle_embed_ocr.ipynb.

Tu kiem tra (khong can GPU, khong can tai model):  python embed_keyframes.py --selftest
"""
import argparse
import csv
import io
import json
import os
import re
import shutil
import sys
import tarfile
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from PIL import Image

DEFAULT_MODEL = 'ViT-SO400M-16-SigLIP2-384'
DEFAULT_PRETRAINED = 'webli'
# Kaggle doc dataset qua o mang: moi lan mo file ~0.2s nhung mo nhieu file cung luc duoc
# -> doc song song bang nhieu thread (doc tuan tu tung anh chi ~5 anh/s, GPU ngoi cho)
IO_THREADS = 32
_io_pool = ThreadPoolExecutor(IO_THREADS)


def _read_file(path):
    with open(path, 'rb') as f:
        return f.read()


def feat_dir_name(model):
    return 'features-' + re.sub(r'[^a-z0-9]+', '-', model.lower()).strip('-')


def list_videos(data_dirs, only=None, shard=None):
    """[(vid, data_dir)] sap theo ten. Trung ten giua cac thu muc thi lay thu muc dau, giong build_index."""
    seen, out = set(), []
    for d in data_dirs:
        mdir = os.path.join(d, 'map-keyframes')
        if not os.path.isdir(mdir):
            raise SystemExit(f'Khong thay map-keyframes trong {d}')
        for f in sorted(os.listdir(mdir)):
            v = f[:-4]
            if not f.endswith('.csv') or v in seen or (only and not any(v.startswith(p) for p in only)):
                continue
            seen.add(v)
            out.append((v, d))
    out.sort()
    if shard:
        i, n = shard
        out = out[i::n]
    return out


def _jpg_key(name):
    stem = os.path.splitext(os.path.basename(name))[0]
    return int(stem) if stem.isdigit() else None


def read_video(vid, d):
    """(frame_idx theo thu tu csv, byte jpg cung thu tu). Thu tu csv = thu tu build_index ghep feature.

    Anh lay tu keyframes/<vid>.tar (output notebook), hoac thu muc da giai nen
    (keyframes/<vid>.tar/ hay keyframes/<vid>/ -- Kaggle co the tu giai nen tar khi upload dataset).
    """
    with open(os.path.join(d, 'map-keyframes', f'{vid}.csv'), newline='') as f:
        frame_idx = [int(r['frame_idx']) for r in csv.DictReader(f)]
    tar = os.path.join(d, 'keyframes', f'{vid}.tar')
    if os.path.isfile(tar):
        # doc ca file 1 lan roi tach trong RAM: tranh nhay coc tung anh tren o mang
        with tarfile.open(fileobj=io.BytesIO(_read_file(tar))) as tf:
            members = {_jpg_key(m.name): m for m in tf.getmembers() if m.isfile() and m.name.endswith('.jpg')}
            missing = [fi for fi in frame_idx if fi not in members]
            if missing:
                raise RuntimeError(f'tar thieu {len(missing)}/{len(frame_idx)} anh, vd frame {missing[:3]}')
            return frame_idx, [tf.extractfile(members[fi]).read() for fi in frame_idx]
    folder = next((p for p in (tar, os.path.join(d, 'keyframes', vid)) if os.path.isdir(p)), None)
    if folder is None:
        raise FileNotFoundError(f'khong thay {tar} (file hoac thu muc) hay keyframes/{vid}/')
    paths = {}
    for root, _, files in os.walk(folder):
        for fn in files:
            if fn.endswith('.jpg') and _jpg_key(fn) is not None:
                paths[_jpg_key(fn)] = os.path.join(root, fn)
    missing = [fi for fi in frame_idx if fi not in paths]
    if missing:
        raise RuntimeError(f'thu muc thieu {len(missing)}/{len(frame_idx)} anh, vd frame {missing[:3]}')
    return frame_idx, list(_io_pool.map(_read_file, [paths[fi] for fi in frame_idx]))


def seed_from(resume_dirs, out_dir, ext):
    """Chep ket qua lan chay truoc (Kaggle: output version cu, them vao lam Input) sang out_dir de bo qua."""
    n = 0
    for rd in resume_dirs or []:
        for fn in os.listdir(rd):
            dst = os.path.join(out_dir, fn)
            if fn.endswith(ext) and not os.path.exists(dst):
                tmp = f'{dst}.{os.getpid()}.part'      # 2 tien trinh --shard cung chep -> khong dung chung file tam
                shutil.copyfile(os.path.join(rd, fn), tmp)
                os.replace(tmp, dst)
                n += 1
    if resume_dirs:
        print(f'resume: chep {n} file {ext} tu {resume_dirs}')


def check_writable(path):
    try:
        os.makedirs(path, exist_ok=True)
        probe = os.path.join(path, f'.write_test_{os.getpid()}')    # nhieu tien trinh (--shard) cung luc
        open(probe, 'w').close()
        os.remove(probe)
    except OSError as e:
        raise SystemExit(f'Khong ghi duoc vao {path} ({e}). Kaggle: dataset chi doc -> dung --out /kaggle/working/...')


def is_done(path, n):
    if not os.path.exists(path):
        return False
    try:
        return np.load(path, mmap_mode='r').shape[0] == n
    except Exception:          # file hong do bi ngat giua chung -> lam lai
        return False


def save_npy(path, arr):
    """Ghi .part roi doi ten: file o dich luon hoan chinh."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.part'
    with open(tmp, 'wb') as f:
        np.save(f, arr)
    os.replace(tmp, path)


def encode_video(raw, load, encode, batch, pool, dim, ahead=3):
    """byte jpg -> [N, D] float16. Giai ma/preprocess song song tren CPU, giu toi da ahead lo cho GPU."""
    futs, it, out = deque(), iter(raw), []

    def fill():
        while len(futs) < batch * ahead:
            b = next(it, None)
            if b is None:
                return
            futs.append(pool.submit(load, b))

    fill()
    while futs:
        x = [futs.popleft().result() for _ in range(min(batch, len(futs)))]
        fill()
        out.append(encode(x))
    return np.concatenate(out).astype(np.float16) if out else np.zeros((0, dim), np.float16)


def load_model(name, pretrained, device=None, fp16=True):
    """(preprocess, encode, dim). Preprocess lay theo pretrained cfg: SigLIP la 'squash' ca khung 16:9,
    khong cat 2 ben nhu center-crop cua ViT-B-32 hien tai."""
    import torch
    import torch.nn.functional as F
    import open_clip

    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    fp16 = fp16 and device == 'cuda'
    model, _, preprocess = open_clip.create_model_and_transforms(name, pretrained=pretrained, device=device)
    model.eval()
    if fp16:
        model = model.half()

    @torch.inference_mode()
    def encode(tensors):
        x = torch.stack(tensors).to(device, non_blocking=True)
        e = model.encode_image(x.half() if fp16 else x)
        return F.normalize(e.float(), dim=-1).cpu().numpy()

    dim = encode([preprocess(Image.new('RGB', (64, 64)))]).shape[1]
    print(f'{name} ({pretrained}) tren {device}{" fp16" if fp16 else ""} | dim {dim}\n  preprocess: '
          + ' -> '.join(type(t).__name__ for t in getattr(preprocess, 'transforms', [])))
    return preprocess, encode, dim


def run(a, preprocess, encode, dim):
    vids = list_videos(a.data, a.only, a.shard)
    out_of = (lambda d: a.out) if a.out else (lambda d: os.path.join(d, feat_dir_name(a.model)))

    for d in sorted({d for _, d in vids}):
        check_writable(out_of(d))
        info = os.path.join(out_of(d), 'info.json')
        if not os.path.exists(info):
            tmp = f'{info}.{os.getpid()}.part'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump({'model': a.model, 'pretrained': a.pretrained, 'dim': dim,
                           'created': time.strftime('%Y-%m-%d %H:%M:%S')}, f, indent=2)
            os.replace(tmp, info)

    if a.out:
        seed_from(a.resume, a.out, '.npy')
    def count(vd):
        with open(os.path.join(vd[1], 'map-keyframes', f'{vd[0]}.csv'), newline='') as f:
            return sum(1 for _ in csv.DictReader(f))

    todo = []
    for (vid, d), n in zip(vids, _io_pool.map(count, vids)):     # dem song song, 726 csv tuan tu mat ~2 phut
        p = os.path.join(out_of(d), f'{vid}.npy')
        if not is_done(p, n):
            todo.append((vid, d, p, n))
    n_all = sum(t[3] for t in todo)
    print(f'{len(vids)} video, con {len(todo)} video / {n_all:,} keyframe chua encode')
    if a.limit:
        todo = todo[:a.limit]

    load = (lambda b: preprocess(Image.open(io.BytesIO(b)).convert('RGB')))
    errors, done_kf, t_all = [], 0, time.time()
    with ThreadPoolExecutor(a.workers) as pool, ThreadPoolExecutor(1) as io_pool:
        # doc tar cua video ke tiep trong luc GPU dang chay video hien tai (Drive doc cham)
        nxt = io_pool.submit(read_video, *todo[0][:2]) if todo else None
        for k, (vid, d, path, n) in enumerate(todo):
            if a.max_hours and time.time() - t_all > a.max_hours * 3600:
                print(f'>>> Da chay {a.max_hours}h -> dung som, con {len(todo) - k} video. Chay lai voi --resume.')
                break
            cur = nxt
            nxt = io_pool.submit(read_video, *todo[k + 1][:2]) if k + 1 < len(todo) else None
            t0 = time.time()
            try:
                _, raw = cur.result()
                emb = encode_video(raw, load, encode, a.batch, pool, dim)
                if len(emb) != n:
                    raise RuntimeError(f'lech so dong: csv={n} emb={len(emb)}')
                save_npy(path, emb)
            except Exception as e:
                errors.append((vid, f'{type(e).__name__}: {e}'))
                print(f'!! {vid}: {errors[-1][1][:300]}')
                continue
            dt = time.time() - t0
            done_kf += n
            speed = done_kf / (time.time() - t_all)
            left = sum(t[3] for t in todo[k + 1:])
            print(f'[{k + 1}/{len(todo)}] {vid}: {n:5d} kf, {n / max(dt, 1e-6):6.1f} anh/s | '
                  f'TB {speed:6.1f} anh/s | ETA {left / max(speed, 1e-6) / 3600:.2f}h')

    print(f'Xong {len(todo) - len(errors)}/{len(todo)} video, {done_kf:,} keyframe trong '
          f'{(time.time() - t_all) / 60:.1f} phut | loi {len(errors)}')
    for vid, msg in errors:
        print('  ', vid, msg[:200])
    feat = a.out or feat_dir_name(a.model)
    print('\nTao index:\n  python build_index.py --data ' + ' '.join(a.data)
          + f' --out <thu muc index moi> --feat-dir {feat} --model {a.model} --pretrained {a.pretrained}')
    return errors


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument('--data', nargs='+', help='thu muc co keyframes/*.tar + map-keyframes/*.csv')
    p.add_argument('--model', default=DEFAULT_MODEL)
    p.add_argument('--pretrained', default=DEFAULT_PRETRAINED)
    p.add_argument('--out', default=None,
                   help='ghi moi .npy vao 1 thu muc nay (mac dinh <data>/features-<model>/). '
                        'Dung --out thi build_index can --feat-dir <duong dan tuyet doi nay>')
    p.add_argument('--batch', type=int, default=64)
    p.add_argument('--workers', type=int, default=min(8, os.cpu_count() or 2), help='so thread giai ma jpg')
    p.add_argument('--device', default=None)
    p.add_argument('--fp32', action='store_true', help='tat fp16 tren GPU')
    p.add_argument('--only', nargs='*', default=None, help='chi lay video co tien to nay, vd L01 L02')
    p.add_argument('--shard', default=None, help='i/n: may thu i (tu 0) trong n may chia nhau')
    p.add_argument('--limit', type=int, default=0, help='chi chay N video dau (do toc do)')
    p.add_argument('--resume', nargs='*', default=None, help='thu muc .npy cua lan chay truoc (chi dung voi --out)')
    p.add_argument('--max-hours', type=float, default=0, help='dung som sau N gio (Kaggle gioi han 12h)')
    p.add_argument('--selftest', action='store_true')
    a = p.parse_args(argv)
    if a.shard:
        i, n = map(int, a.shard.split('/'))
        assert 0 <= i < n, '--shard phai dang i/n voi 0 <= i < n'
        a.shard = (i, n)
    if not a.selftest and not a.data:
        p.error('can --data')
    return a


def main():
    a = parse_args()
    if a.selftest:
        return selftest()
    if a.out:
        a.out = os.path.abspath(a.out)
        check_writable(a.out)
    pre, enc, dim = load_model(a.model, a.pretrained, a.device, not a.fp32)
    run(a, pre, enc, dim)


def selftest():
    """Du lieu gia giong output notebook + encoder gia (mau trung binh) -> kiem tra thu tu, resume,
    shard, video loi khong lam dung ca lo, va build_index doc duoc ket qua."""
    import shutil
    import subprocess
    import tempfile

    def make_batch(root, vids):
        for sub in ('keyframes', 'map-keyframes'):
            os.makedirs(os.path.join(root, sub), exist_ok=True)
        for vid, fis in vids.items():
            tmp = os.path.join(root, '_kf', vid)
            os.makedirs(tmp)
            for fi in reversed(fis):           # ghi tar nguoc thu tu -> phai doc theo csv, khong theo tar
                Image.new('RGB', (64, 36), color_of(vid, fi)).save(os.path.join(tmp, f'{fi:06d}.jpg'), quality=95)
            with tarfile.open(os.path.join(root, 'keyframes', f'{vid}.tar'), 'w') as tf:
                tf.add(tmp, arcname=vid)
            with open(os.path.join(root, 'map-keyframes', f'{vid}.csv'), 'w', newline='') as f:
                w = csv.writer(f)
                w.writerow(['n', 'pts_time', 'fps', 'frame_idx'])
                for n, fi in enumerate(fis, 1):
                    w.writerow([n, fi / 25, 25.0, fi])
        shutil.rmtree(os.path.join(root, '_kf'))

    def color_of(vid, fi):
        return (int(vid[-1]) * 40 % 256, fi % 256, 200)

    pre = lambda im: np.asarray(im.resize((8, 8)), np.float32)
    calls = []

    def enc(xs):
        calls.append(len(xs))
        m = np.stack([x.mean(axis=(0, 1)) for x in xs])
        return m / np.linalg.norm(m, axis=1, keepdims=True)

    with tempfile.TemporaryDirectory() as tmp:
        b1, b2 = os.path.join(tmp, 'Batch1'), os.path.join(tmp, 'Batch2')
        make_batch(b1, {'L01_V001': [0, 12, 50, 99, 130], 'L01_V002': [3, 7]})
        make_batch(b2, {'L02_V001': list(range(0, 400, 4)), 'L01_V002': [1]})   # trung ten -> lay Batch1

        a = parse_args(['--data', b1, b2, '--batch', '16', '--workers', '2'])
        assert run(a, pre, enc, 3) == []
        assert max(calls) <= 16 and sum(calls) == 5 + 2 + 100, calls
        fd = feat_dir_name(DEFAULT_MODEL)
        e1 = np.load(os.path.join(b1, fd, 'L01_V001.npy'))
        assert e1.dtype == np.float16 and e1.shape == (5, 3), e1.shape
        exp = enc([pre(Image.new('RGB', (64, 36), color_of('L01_V001', fi))) for fi in [0, 12, 50, 99, 130]])
        assert np.abs(e1.astype(np.float32) - exp).max() < 0.02, 'sai thu tu so voi csv'
        assert not os.path.exists(os.path.join(b2, fd, 'L01_V002.npy')), 'video trung ten phai lay Batch1'
        assert json.load(open(os.path.join(b1, fd, 'info.json')))['dim'] == 3

        # resume: lan 2 khong encode gi; xoa 1 file / lam hong 1 file -> chi lam lai dung file do
        calls.clear()
        run(a, pre, enc, 3)
        assert calls == [], calls
        os.remove(os.path.join(b1, fd, 'L01_V002.npy'))
        open(os.path.join(b2, fd, 'L02_V001.npy'), 'wb').write(b'hong')
        run(a, pre, enc, 3)
        assert sum(calls) == 2 + 100, calls
        assert not any(f.endswith('.part') for d in (b1, b2) for f in os.listdir(os.path.join(d, fd)))

        # shard: 2 may chia nhau, khong trung, du ca bo
        s0 = [v for v, _ in list_videos([b1, b2], shard=(0, 2))]
        s1 = [v for v, _ in list_videos([b1, b2], shard=(1, 2))]
        assert sorted(s0 + s1) == ['L01_V001', 'L01_V002', 'L02_V001'] and not set(s0) & set(s1)

        # video thieu anh trong tar -> bao loi video do, cac video khac van chay
        with open(os.path.join(b1, 'map-keyframes', 'L01_V001.csv'), 'a', newline='') as f:
            csv.writer(f).writerow([6, 9.0, 25.0, 225])
        errs = run(a, pre, enc, 3)
        assert len(errs) == 1 and errs[0][0] == 'L01_V001' and 'thieu' in errs[0][1], errs
        with open(os.path.join(b1, 'map-keyframes', 'L01_V001.csv'), newline='') as f:
            rows = list(csv.reader(f))[:-1]
        with open(os.path.join(b1, 'map-keyframes', 'L01_V001.csv'), 'w', newline='') as f:
            csv.writer(f).writerows(rows)

        # --out: gom 1 thu muc; build_index doc duoc bang --feat-dir tuyet doi
        out = os.path.join(tmp, 'feat_all')
        run(parse_args(['--data', b1, b2, '--out', out, '--batch', '8']), pre, enc, 3)
        assert sorted(os.listdir(out)) == ['L01_V001.npy', 'L01_V002.npy', 'L02_V001.npy', 'info.json']
        bi = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'build_index.py')
        if os.path.exists(bi):         # Kaggle chi co 2 script -> bo qua buoc nay
            idx = os.path.join(tmp, 'index')
            r = subprocess.run([sys.executable, bi, '--data', b1, b2, '--out', idx, '--feat-dir', out,
                                '--model', DEFAULT_MODEL, '--pretrained', DEFAULT_PRETRAINED],
                               capture_output=True, text=True)
            assert r.returncode == 0, r.stderr
            feats = np.load(os.path.join(idx, 'features.npy'))
            assert feats.shape == (107, 3), feats.shape
            assert '!! bo qua' not in r.stdout or 'trung ten' in r.stdout, r.stdout

        # Kaggle tu giai nen tar: keyframes/<vid>.tar/<vid>/*.jpg va keyframes/<vid>/*.jpg -> van doc dung thu tu
        fis, raw = read_video('L01_V001', b1)
        for layout in ('tar_dir', 'plain_dir'):
            kd = os.path.join(tmp, layout, 'keyframes')
            shutil.copytree(os.path.join(b1, 'map-keyframes'), os.path.join(tmp, layout, 'map-keyframes'))
            dst = os.path.join(kd, 'L01_V001.tar') if layout == 'tar_dir' else kd
            with tarfile.open(os.path.join(b1, 'keyframes', 'L01_V001.tar')) as tf:
                tf.extractall(dst)
            fis2, raw2 = read_video('L01_V001', os.path.join(tmp, layout))
            assert fis2 == fis and raw2 == raw, layout

        # o mang cham (moi file 0.05s): 100 anh tuan tu mat 5s, song song phai < 1s va dung thu tu
        global _read_file
        orig = _read_file

        def slow(path):
            time.sleep(0.05)
            return orig(path)
        _read_file = slow
        try:
            plain = os.path.join(tmp, 'plain_dir2')
            shutil.copytree(os.path.join(b2, 'map-keyframes'), os.path.join(plain, 'map-keyframes'))
            with tarfile.open(os.path.join(b2, 'keyframes', 'L02_V001.tar')) as tf:
                tf.extractall(os.path.join(plain, 'keyframes'))
            t0 = time.time()
            fis3, raw3 = read_video('L02_V001', plain)
            dt = time.time() - t0
        finally:
            _read_file = orig
        assert dt < 1.0, f'doc song song qua cham: {dt:.2f}s'
        assert (fis3, raw3) == read_video('L02_V001', b2), 'doc song song sai thu tu'

        # resume: output lan truoc -> chep sang --out moi, khong encode lai
        out2 = os.path.join(tmp, 'feat_run2')
        calls.clear()
        run(parse_args(['--data', b1, b2, '--out', out2, '--resume', out]), pre, enc, 3)
        assert calls == [] and sorted(f for f in os.listdir(out2) if f.endswith('.npy')) == \
            ['L01_V001.npy', 'L01_V002.npy', 'L02_V001.npy'], calls

        # max-hours rat nho -> dung ngay, khong loi
        out3 = os.path.join(tmp, 'feat_run3')
        a3 = parse_args(['--data', b1, '--out', out3, '--max-hours', '1e-9'])
        assert run(a3, pre, enc, 3) == [] and not [f for f in os.listdir(out3) if f.endswith('.npy')]
    print('embed_keyframes.py selftest OK')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile ocr_keyframes.py
"""OCR chu tren man hinh cho keyframe DA TRICH (doc thang tar), EasyOCR tieng Viet tren GPU.

Dau ra: <out>/<vid>.jsonl, moi dong 1 keyframe CO CHU (keyframe khong co chu thi khong ghi):
    {"frame_idx": 12345, "text": "GIAI PHONG KHI HIDRO ...", "lines": [{"text": "...", "box": [x1, y1, x2, y2], "conf": 0.93}]}
Video khong co chu nao van co file rong = da xong (de chay tiep khong lam lai).

Dung trong engine: chep thu muc <out> vao <data>/ocr/, hoac chay app.py --ocr <out>.
Engine so khop KHONG DAU (norm_vi) nen OCR sai dau it anh huong; quan trong la bat duoc chu.

    pip install easyocr
    python ocr_keyframes.py --data .../Batch1 .../Batch2 --out ocr --limit 2      # do toc do
    python ocr_keyframes.py --data .../Batch1 .../Batch2 --out ocr
Kaggle: --out /kaggle/working/ocr, --resume <output lan truoc>, --max-hours 11. Xem kaggle_embed_ocr.ipynb.
Chu nho (ticker, bien hieu) bi sot: thu --mag 1.5 (detector phong to anh, cham hon ~2 lan).

Tu kiem tra (khong can GPU/model):  python ocr_keyframes.py --selftest
"""
import argparse
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

from embed_keyframes import check_writable, list_videos, read_video, seed_from


def jpeg_size(b):
    """(w, h) cua jpg tu header, khong giai ma ca anh."""
    import struct
    i = 2
    while i < len(b):
        if b[i] != 0xFF:
            i += 1
            continue
        marker = b[i + 1]
        if marker in (0xC0, 0xC1, 0xC2):
            h, w = struct.unpack('>HH', b[i + 5:i + 9])
            return w, h
        i += 2 + struct.unpack('>H', b[i + 2:i + 4])[0]
    raise ValueError('khong doc duoc kich thuoc jpg')


def to_lines(res, min_conf):
    """Ket qua EasyOCR [(4 diem, text, conf)] -> dong da loc, sap tren xuong, trai qua phai."""
    out = []
    for pts, text, conf in res:
        text = ' '.join(str(text).split())
        if not text or conf < min_conf or (len(text) == 1 and not text.isalnum()):
            continue
        xs, ys = [float(p[0]) for p in pts], [float(p[1]) for p in pts]
        out.append({'text': text, 'box': [round(min(xs)), round(min(ys)), round(max(xs)), round(max(ys))],
                    'conf': round(float(conf), 3)})
    # cung hang neu tam y lech < nua chieu cao dong -> gom hang theo bac 12px roi xet x
    out.sort(key=lambda l: (round((l['box'][1] + l['box'][3]) / 2 / 12), l['box'][0]))
    return out


def ocr_video(raw, frame_idx, reader_fn, batch):
    """-> list dong jsonl (chi keyframe co chu). reader_fn(list byte jpg, (w, h)) -> list ket qua tung anh."""
    rows = []
    for s in range(0, len(raw), batch):
        chunk, fis = raw[s:s + batch], frame_idx[s:s + batch]
        size = jpeg_size(chunk[0])
        for fi, lines in zip(fis, reader_fn(chunk, size)):
            if lines:
                rows.append({'frame_idx': fi, 'text': ' '.join(l['text'] for l in lines), 'lines': lines})
    return rows


def save_jsonl(path, rows):
    tmp = path + '.part'
    with open(tmp, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    os.replace(tmp, path)


def make_reader(langs, mag, min_conf, rec_batch, text_threshold, gpu=True):
    import easyocr
    reader = easyocr.Reader(langs, gpu=gpu, verbose=False)

    def fn(chunk, size):
        w, h = size
        # cung kich thuoc trong 1 lo (1 video) -> detector CRAFT chay theo lo tren GPU
        res = reader.readtext_batched(list(chunk), n_width=w, n_height=h, batch_size=rec_batch,
                                      mag_ratio=mag, text_threshold=text_threshold, paragraph=False)
        return [to_lines(r, min_conf) for r in res]

    print(f'EasyOCR {langs} | gpu={gpu} | mag {mag} | min_conf {min_conf}')
    return fn


def run(a, reader_fn):
    vids = list_videos(a.data, a.only, a.shard)
    check_writable(a.out)
    seed_from(a.resume, a.out, '.jsonl')
    todo = [(v, d) for v, d in vids if not os.path.exists(os.path.join(a.out, f'{v}.jsonl'))]
    print(f'{len(vids)} video, con {len(todo)} video chua OCR')
    if a.limit:
        todo = todo[:a.limit]

    errors, done_kf, t_all = [], 0, time.time()
    with ThreadPoolExecutor(1) as io_pool:
        nxt = io_pool.submit(read_video, *todo[0]) if todo else None
        for k, (vid, d) in enumerate(todo):
            if a.max_hours and time.time() - t_all > a.max_hours * 3600:
                print(f'>>> Da chay {a.max_hours}h -> dung som, con {len(todo) - k} video. Chay lai voi --resume.')
                break
            cur = nxt
            nxt = io_pool.submit(read_video, *todo[k + 1]) if k + 1 < len(todo) else None
            t0 = time.time()
            try:
                frame_idx, raw = cur.result()
                rows = ocr_video(raw, frame_idx, reader_fn, a.batch)
                save_jsonl(os.path.join(a.out, f'{vid}.jsonl'), rows)
            except Exception as e:
                errors.append((vid, f'{type(e).__name__}: {e}'))
                print(f'!! {vid}: {errors[-1][1][:300]}')
                continue
            dt = time.time() - t0
            done_kf += len(raw)
            speed = done_kf / (time.time() - t_all)
            left_v = len(todo) - k - 1
            avg_kf = done_kf / (k + 1 - len(errors)) if k + 1 > len(errors) else 0
            print(f'[{k + 1}/{len(todo)}] {vid}: {len(raw):5d} kf, {len(rows):5d} co chu, '
                  f'{len(raw) / max(dt, 1e-6):5.1f} anh/s | TB {speed:5.1f} anh/s | '
                  f'ETA ~{left_v * avg_kf / max(speed, 1e-6) / 3600:.2f}h')
    print(f'Xong {len(todo) - len(errors)}/{len(todo)} video, {done_kf:,} keyframe trong '
          f'{(time.time() - t_all) / 60:.1f} phut | loi {len(errors)}')
    for vid, msg in errors:
        print('  ', vid, msg[:200])
    return errors


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument('--data', nargs='+', help='thu muc co keyframes/ + map-keyframes/')
    p.add_argument('--out', default='ocr', help='thu muc ghi <vid>.jsonl')
    p.add_argument('--langs', nargs='+', default=['vi'], help='ngon ngu EasyOCR (vi da gom chu Latin/Anh)')
    p.add_argument('--batch', type=int, default=16, help='so anh / lo detector')
    p.add_argument('--rec-batch', type=int, default=64, help='so dong chu / lo recognizer')
    p.add_argument('--mag', type=float, default=1.0, help='phong to anh cho detector (1.5: bat chu nho hon)')
    p.add_argument('--text-threshold', type=float, default=0.7)
    p.add_argument('--min-conf', type=float, default=0.2, help='bo dong chu co do tin cay thap hon')
    p.add_argument('--cpu', action='store_true')
    p.add_argument('--only', nargs='*', default=None, help='chi lay video co tien to nay, vd L01 L02')
    p.add_argument('--shard', default=None, help='i/n: may thu i (tu 0) trong n may chia nhau')
    p.add_argument('--limit', type=int, default=0, help='chi chay N video dau (do toc do)')
    p.add_argument('--resume', nargs='*', default=None, help='thu muc .jsonl cua lan chay truoc')
    p.add_argument('--max-hours', type=float, default=0, help='dung som sau N gio (Kaggle gioi han 12h)')
    p.add_argument('--selftest', action='store_true')
    a = p.parse_args(argv)
    if a.shard:
        i, n = map(int, a.shard.split('/'))
        assert 0 <= i < n
        a.shard = (i, n)
    if not a.selftest and not a.data:
        p.error('can --data')
    a.out = os.path.abspath(a.out)
    return a


def main():
    a = parse_args()
    if a.selftest:
        return selftest()
    run(a, make_reader(a.langs, a.mag, a.min_conf, a.rec_batch, a.text_threshold, gpu=not a.cpu))


def selftest():
    """Reader gia -> kiem tra dinh dang dau ra, loc/sap dong, resume, va engine doc duoc file OCR."""
    import csv
    import io
    import shutil
    import tarfile
    import tempfile

    import numpy as np
    from PIL import Image

    # 1. to_lines: loc conf thap / ky tu rac, sap tren xuong trai qua phai
    res = [([[300, 400], [500, 400], [500, 430], [300, 430]], 'Hà Nội', 0.9),
           ([[10, 402], [200, 402], [200, 428], [10, 428]], 'THỜI SỰ', 0.95),
           ([[10, 20], [90, 20], [90, 40], [10, 40]], 'VTV1', 0.8),
           ([[0, 0], [5, 0], [5, 5], [0, 5]], '|', 0.9),
           ([[0, 0], [50, 0], [50, 9], [0, 9]], 'mo', 0.05)]
    assert [l['text'] for l in to_lines(res, 0.2)] == ['VTV1', 'THỜI SỰ', 'Hà Nội']
    assert to_lines(res, 0.2)[0]['box'] == [10, 20, 90, 40]

    buf = io.BytesIO()
    Image.new('RGB', (854, 480)).save(buf, 'JPEG')
    assert jpeg_size(buf.getvalue()) == (854, 480)

    with tempfile.TemporaryDirectory() as tmp:
        b1 = os.path.join(tmp, 'Batch1')
        for sub in ('keyframes', 'map-keyframes'):
            os.makedirs(os.path.join(b1, sub))
        vids = {'L01_V001': [0, 50, 100], 'L01_V002': [10, 20]}
        for vid, fis in vids.items():
            kd = os.path.join(tmp, '_kf', vid)
            os.makedirs(kd)
            for fi in fis:
                Image.new('RGB', (64, 36), (fi % 256, 0, 0)).save(os.path.join(kd, f'{fi:06d}.jpg'))
            with tarfile.open(os.path.join(b1, 'keyframes', f'{vid}.tar'), 'w') as tf:
                tf.add(kd, arcname=vid)
            with open(os.path.join(b1, 'map-keyframes', f'{vid}.csv'), 'w', newline='') as f:
                w = csv.writer(f)
                w.writerow(['n', 'pts_time', 'fps', 'frame_idx'])
                w.writerows([[n + 1, fi / 25, 25.0, fi] for n, fi in enumerate(fis)])

        calls = []

        def fake(chunk, size):
            calls.append((len(chunk), size))
            out = []
            for b in chunk:
                red = int(np.asarray(Image.open(io.BytesIO(b)))[0, 0, 0])
                # anh frame 50 co chu, cac anh khac khong
                out.append(to_lines([([[0, 0], [9, 0], [9, 9], [0, 9]], f'GIẢI PHÓNG {red}', 0.9)], 0.2)
                           if 45 <= red <= 55 else [])
            return out

        out = os.path.join(tmp, 'ocr')
        a = parse_args(['--data', b1, '--out', out, '--batch', '2'])
        assert run(a, fake) == []
        assert calls[0] == (2, (64, 36)) and sum(c[0] for c in calls) == 5
        rows = [json.loads(l) for l in open(os.path.join(out, 'L01_V001.jsonl'), encoding='utf-8')]
        assert len(rows) == 1 and rows[0]['frame_idx'] == 50 and rows[0]['text'].startswith('GIẢI PHÓNG')
        assert open(os.path.join(out, 'L01_V002.jsonl')).read() == ''          # khong chu van danh dau xong

        calls.clear()
        run(a, fake)
        assert calls == [], 'lan 2 phai bo qua video da xong'
        out2 = os.path.join(tmp, 'ocr2')
        run(parse_args(['--data', b1, '--out', out2, '--resume', out]), fake)
        assert calls == [] and sorted(os.listdir(out2)) == ['L01_V001.jsonl', 'L01_V002.jsonl']

        # engine doc duoc: dat vao <data>/ocr roi loc khong dau (Kaggle khong co engine/build_index -> bo qua)
        here = os.path.dirname(os.path.abspath(__file__))
        if not all(os.path.exists(os.path.join(here, f)) for f in ('build_index.py', 'engine.py')):
            print('ocr_keyframes.py selftest OK (bo qua kiem tra engine)')
            return
        shutil.copytree(out, os.path.join(b1, 'ocr'))
        sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
        import subprocess
        idx = os.path.join(tmp, 'index')
        os.makedirs(os.path.join(b1, 'features-vit-b-32'))
        for vid, fis in vids.items():
            np.save(os.path.join(b1, 'features-vit-b-32', f'{vid}.npy'), np.eye(len(fis), 4, dtype=np.float16))
        r = subprocess.run([sys.executable, os.path.join(here, 'build_index.py'), '--data', b1, '--out', idx],
                           capture_output=True, text=True)
        assert r.returncode == 0, r.stderr
        try:
            from engine import SearchEngine
        except ImportError:          # may khong co torch: bo qua buoc nay
            print('  (bo qua kiem tra engine: thieu torch)')
        else:
            eng = SearchEngine(idx, load_model=False)
            res = eng.search('', ocr='giai phong')
            assert [(x['video'], x['frame_idx']) for x in res] == [('L01_V001', 50)], res
    print('ocr_keyframes.py selftest OK')


if __name__ == '__main__':
    main()


In [ ]:
# 5. Tim du lieu + tu kiem tra script
import os, glob

def find_batches(root='/kaggle/input'):
    out = []
    for dp, dns, _ in os.walk(root):
        if 'map-keyframes' in dns:
            out.append(dp)
        dns[:] = [d for d in dns if d not in ('keyframes', 'map-keyframes') and not d.startswith('features-')]
    return sorted(out)

DATA = DATA_DIRS or find_batches()
assert DATA, 'Khong thay thu muc nao co map-keyframes/ trong /kaggle/input -> kiem tra Add Input'
for d in DATA:
    kf = os.path.join(d, 'keyframes')
    items = os.listdir(kf) if os.path.isdir(kf) else []
    n_tar = sum(f.endswith('.tar') and os.path.isfile(os.path.join(kf, f)) for f in items)
    print(f'{d}: {len(glob.glob(d + "/map-keyframes/*.csv"))} csv | keyframes/: {n_tar} tar, '
          f'{len(items) - n_tar} thu muc/khac')

!python embed_keyframes.py --selftest 2>&1 | tail -1
!python ocr_keyframes.py --selftest 2>&1 | tail -1
if RUN_OCR:   # tai model EasyOCR 1 lan truoc, tranh nhieu tien trinh cung ghi 1 file model
    import subprocess, sys
    subprocess.run([sys.executable, '-c', "import easyocr; easyocr.Reader(['vi'], gpu=False, verbose=False); print('EasyOCR model OK')"], check=True)

In [ ]:
# 6. Chay: chia video cho N_NOTEBOOKS notebook x GPU x tien trinh, tat ca chay cung luc
import subprocess, sys, time, torch

n_gpu = torch.cuda.device_count()
gpus = list(range(max(n_gpu, 1)))
if RUN_EMBED and RUN_OCR:
    emb_gpus, ocr_gpus = gpus[:1], (gpus[1:] or gpus[:1])
else:
    emb_gpus, ocr_gpus = gpus, gpus
parallel = not (RUN_EMBED and RUN_OCR and len(gpus) < 2)    # 1 GPU ma bat ca hai -> lan luot

def shard_args(k, per_nb):
    """Shard toan cuc: notebook NOTEBOOK_ID lay cac phan NOTEBOOK_ID*per_nb .. +per_nb-1."""
    n = N_NOTEBOOKS * per_nb
    return [] if n == 1 else ['--shard', f'{NOTEBOOK_ID * per_nb + k}/{n}']

jobs = []   # (ten, gpu, lenh)
if RUN_EMBED:
    for k, g in enumerate(emb_gpus):
        jobs.append((f'embed{k}', g, [sys.executable, 'embed_keyframes.py', '--data', *DATA, '--out', OUT_FEAT,
                     '--model', EMBED_MODEL, '--pretrained', EMBED_PRETRAINED, '--batch', str(EMBED_BATCH)]
                     + shard_args(k, len(emb_gpus)) + (['--resume', *RESUME_FEAT] if RESUME_FEAT else [])))
if RUN_OCR:
    slots = [g for g in ocr_gpus for _ in range(OCR_PROCS_PER_GPU)]
    for k, g in enumerate(slots):
        jobs.append((f'ocr{k}', g, [sys.executable, 'ocr_keyframes.py', '--data', *DATA, '--out', OUT_OCR,
                     '--mag', str(OCR_MAG), '--batch', str(OCR_BATCH)]
                     + shard_args(k, len(slots)) + (['--resume', *RESUME_OCR] if RESUME_OCR else [])))
hours = MAX_HOURS if parallel else MAX_HOURS / 2
for _, _, cmd in jobs:
    cmd += ['--max-hours', str(hours)] + (['--limit', str(LIMIT)] if LIMIT else [])
os.makedirs('/kaggle/working/logs', exist_ok=True)

def tail(name, n=1):
    p = f'/kaggle/working/logs/{name}.log'
    if not os.path.exists(p):
        return []
    lines = open(p, encoding='utf-8', errors='replace').read().splitlines()
    return [l for l in lines if l.strip() and 'Warning' not in l and 'warn(' not in l][-n:]

def launch(name, cmd, gpu, n_par):
    print(f'[{name}] GPU {gpu}: {" ".join(cmd)}')
    # chia deu CPU cho cac tien trinh, tranh moi tien trinh tu mo du so thread roi tranh nhau
    threads = str(max(1, (os.cpu_count() or 4) // n_par))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1',
               OMP_NUM_THREADS=threads, MKL_NUM_THREADS=threads)
    return subprocess.Popen(cmd, stdout=open(f'/kaggle/working/logs/{name}.log', 'w'),
                            stderr=subprocess.STDOUT, env=env)

def wait(procs):
    t0, last = time.time(), 0
    while any(p.poll() is None for _, p in procs):
        time.sleep(15)
        if time.time() - last >= (60 if time.time() - t0 < 600 else 600):
            last = time.time()
            for name, _ in procs:
                print(f'[{(time.time() - t0) / 60:6.1f} phut] {name}: {" ".join(tail(name))[:200]}')
    for name, p in procs:
        print(f'\n===== {name} (exit {p.returncode}) =====')
        print('\n'.join(tail(name, 6)))

print(f'notebook {NOTEBOOK_ID + 1}/{N_NOTEBOOKS} | {n_gpu} GPU | {len(jobs)} tien trinh | {os.cpu_count()} CPU')
if parallel:
    wait([(n, launch(n, c, g, len(jobs))) for n, g, c in jobs])
else:
    for prefix in ('embed', 'ocr'):
        grp = [(n, g, c) for n, g, c in jobs if n.startswith(prefix)]
        if grp:
            wait([(n, launch(n, c, g, len(grp))) for n, g, c in grp])

In [ ]:
# 7. Nen ket qua de tai ve + kiem tra notebook nay da lam du phan cua minh chua
import shutil
sys.path.insert(0, '.')
from embed_keyframes import list_videos

def assigned(prefix):
    """Cac video ma tien trinh <prefix>* cua notebook nay phai lam (theo --shard)."""
    vids = set()
    for name, _, cmd in jobs:
        if name.startswith(prefix):
            sh = tuple(map(int, cmd[cmd.index('--shard') + 1].split('/'))) if '--shard' in cmd else None
            vids |= {v for v, _ in list_videos(DATA, shard=sh)}
    return vids

for prefix, d, name, ext in (('embed', OUT_FEAT, 'siglip2_features', '.npy'), ('ocr', OUT_OCR, 'ocr', '.jsonl')):
    want = assigned(prefix)
    if not want or not os.path.isdir(d):
        continue
    have = {f[:-len(ext)] for f in os.listdir(d) if f.endswith(ext)}
    miss = sorted(want - have)
    z = shutil.make_archive(f'/kaggle/working/{name}_part{NOTEBOOK_ID}', 'zip', d)
    print(f'{name}: {len(want) - len(miss)}/{len(want)} video cua notebook nay -> {z} ({os.path.getsize(z) / 1e6:.0f} MB)')
    if miss:
        print(f'   << CHUA XONG {len(miss)} video (vd {miss[:3]}): chay lai notebook nay voi '
              f'RESUME = ["/kaggle/input/<output nay>/{os.path.basename(d)}"]')
if os.path.isdir(OUT_OCR):
    n_txt = sum(sum(1 for _ in open(f, encoding='utf-8')) for f in glob.glob(OUT_OCR + '/*.jsonl'))
    print(f'OCR: {n_txt:,} keyframe co chu')